# CVAI · E5 top 3 → MLP có ngữ cảnh → điểm theo trọng số
Chọn GPU T4 và tải `cvai_retrieval_r2.zip` ở ô đầu. Nếu đã có kết quả R1 trên Drive, có thể nhập đường dẫn thư mục model để tái sử dụng encoder. Nếu để trống, notebook tự train R1 trước.

In [ ]:
from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys, tempfile, zipfile
drive.mount('/content/drive')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Chọn một file cvai_retrieval_r2.zip')
work = Path(tempfile.mkdtemp(dir='/content'))
with zipfile.ZipFile(next(iter(uploaded))) as archive:
    archive.extractall(work)
os.chdir(work)
if not Path('improve_retrieval_mlp.py').is_file():
    raise ValueError('Gói ZIP thiếu script R2')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## Train trên R1 và chọn bằng validation
Bản mới dùng lại encoder của bản đối chứng. Cả hai nhận cùng top 3. MLP mới đọc thêm loại yêu cầu, các dấu hiệu trong đoạn và tiêu đề mục chứa đoạn đó. Chỉ chọn bản mới khi macro-F1 tăng và sai số điểm trung bình trên validation không tăng.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU trong Runtime / Change runtime type')
def run_script(script, *args):
    subprocess.run([sys.executable, '-u', script, *map(str, args), '--device', 'cuda'], check=True)

base_path = '' # @param {type:"string"}
run = Path('/content/drive/MyDrive/FutureCV-ML') / datetime.now(timezone.utc).strftime('retrieval-r2-%Y%m%dT%H%M%S%fZ')
run.mkdir(parents=True)
if base_path.strip():
    base = Path(base_path.strip())
else:
    baseline_run = run/'r1'
    run_script('train_retrieval_mlp.py', '--data-dir', 'data', '--pdf-dir', 'pdf', '--output-dir', baseline_run)
    base = baseline_run / json.loads((baseline_run/'comparison.json').read_text())['selected']
candidate = run/'context-mlp'
run_script('improve_retrieval_mlp.py', '--base-dir', base, '--data-dir', 'data', '--output-dir', candidate)
comparison = json.loads((candidate/'comparison.json').read_text())
selected = candidate if comparison['selected'] == 'candidate' else base
print('Model đã chọn:', selected)
for name in ('previous', 'candidate'):
    print(name, {k:round(comparison[name][k], 4) for k in ('macro_f1','span_recall_at_3','score_mae')})

## Kiểm tra bằng PDF
Chạy cả hai model trên cùng PDF. Các file JSON giữ nhãn dự đoán, nhãn tham chiếu và đoạn được truy xuất cho từng yêu cầu. Hai tập PDF đều là dữ liệu tổng hợp tiếng Anh.

In [ ]:
for dataset in ('pdf', 'pdf_followup'):
    for name, model in [('previous',base), ('candidate',candidate)]:
        output = run/f'{dataset}_{name}.json'
        run_script('evaluate_jake.py', '--artifact-dir',model,'--pdf-dir',dataset,'--output',output)
        result = json.loads(output.read_text())
        print(dataset, name, {k:round(result[k], 4) for k in ('accuracy','macro_f1','span_recall_at_3','score_mae')})
print('Kết quả trong Drive:', run)